# Compare the original and the compressed AMASS

Compares the raw sequences (`amass_raw` in `config/paths.yaml`) with the compressed ones (`output_dir` of `config/compress.yaml`, written by `scripts/preprocess/compress_amass.py`), dataset by dataset.

1. **Overview:** sequences, frames and hours, and sequences that exist in only one folder.
2. **Comparison** of the selected datasets:
   - *integrity*: the compressed poses are exactly the original rows at `frame_index`;
   - *recall*: distance from a dropped original frame to its nearest kept pose (degrees per joint, 19 joints), against the same number of random frames;
   - *W1*: per-joint Wasserstein distance of the pose distributions, with a noise floor (two independent samples of the original);
   - *speed kept*: the 99th percentile of the compressed joint and root speed as a share of the original. At a low frame rate fast motion is smoothed, so this is below 100%.

Needs `pip install -e ".[dev]"` (adds `ipywidgets`).

In [ ]:
import json
import os

import ipywidgets as w
from IPython.display import display
from omegaconf import OmegaConf

from hp.data import compare as cp
from hp.utils.paths import REPO_ROOT, get_path

raw_dir = get_path("amass_raw")
comp_dir = os.path.expanduser(OmegaConf.load(REPO_ROOT / "config" / "compress.yaml").output_dir)
datasets = sorted(d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, d)))

with open(os.path.join(comp_dir, "compression.json")) as f:
    params = json.load(f)["params"]
print(f"original:   {raw_dir}")
print(f"compressed: {comp_dir}")
print(f"parameters: {params}")

## Overview

In [ ]:
present = [d for d in datasets if os.path.isdir(os.path.join(comp_dir, d))]
display(w.HTML(cp.overview_table(cp.overview(raw_dir, comp_dir, present))))

## Comparison

In [ ]:
selection = w.SelectMultiple(options=present, value=[d for d in ("SSM", "TCDHands") if d in present], rows=10, description="Datasets")
n_ref = w.IntSlider(value=2000, min=500, max=10000, step=500, description="Recall refs", continuous_update=False)
n_w = w.IntSlider(value=1000, min=200, max=3000, step=200, description="W1 samples", continuous_update=False)
n_seq = w.IntSlider(value=100, min=20, max=500, step=20, description="Speed seqs", continuous_update=False)
n_verify = w.IntSlider(value=20, min=5, max=200, step=5, description="Verify seqs", continuous_update=False)
run_btn = w.Button(description="Compute", button_style="primary")
status = w.HTML()
table = w.HTML()
detail = w.Dropdown(description="Details")
detail_img = w.Image(format="png", layout=w.Layout(width="720px"))
results = {}


def on_compute(_):
    results.clear()
    for k, ds in enumerate(selection.value):
        status.value = f"Computing {ds} ({k + 1}/{len(selection.value)}) ..."
        rep = cp.dataset_report(raw_dir, comp_dir, ds, n_ref.value, n_w.value, n_seq.value, n_verify.value)
        results[ds] = (rep, cp.plot_dataset_report(rep))
    status.value = "Done."
    table.value = cp.report_table([r for r, _ in results.values()])
    detail.options = list(results)
    detail.value = next(iter(results))
    show_detail()


def show_detail(change=None):
    if detail.value in results:
        detail_img.value = results[detail.value][1]


run_btn.on_click(on_compute)
detail.observe(show_detail, "value")
w.VBox(layout=w.Layout(align_items="flex-start"), children=[
    selection, n_ref, n_w, n_seq, n_verify, run_btn, status, table, detail, detail_img,
])